# ToxicGuard V5.22 — İroni/Sarkazm Modeli Eğitimi
**Stage 2 modeli: SemEval-2018 Task 3 + SARC Reddit**

Bu notebook, ToxicGuard Two-Stage Cascade mimarisinin ikinci aşaması olan
ironi/sarkazm dedektörünü eğitir.

- **Base model:** `roberta-base`
- **Görev:** Binary sınıflandırma (irony / non_irony)
- **Veri:** SemEval-2018 Task 3A (~3,833) + SARC Reddit (~28,301)
- **Tahmini süre:** T4 GPU ile 25-35 dakika

In [ ]:
# ─── 1. KURULUM ───────────────────────────────────────────────
!pip install -q transformers datasets scikit-learn pandas numpy torch

In [ ]:
# ─── 2. KÜTÜPHANELER ──────────────────────────────────────────
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─── 3. VERİ İNDİRME ─────────────────────────────────────────
# SemEval-2018 Task 3A — GitHub'dan direkt indir
!wget -q -O semeval_train.txt 'https://raw.githubusercontent.com/Cyvhee/SemEval2018-Task3/master/datasets/train/SemEval2018-T3-train-taskA.txt'
!wget -q -O semeval_test.txt  'https://raw.githubusercontent.com/Cyvhee/SemEval2018-Task3/master/datasets/goldtest_TaskA/SemEval2018-T3_gold_test_taskA_emoji.txt'

print('SemEval dosyaları indirildi.')
!head -3 semeval_train.txt

In [ ]:
# ─── 4. SARC KAGGLE DOSYASINI YÜKLEYİN ───────────────────────
# Bu hücreyi çalıştırın, 'train-balanced-sarcasm.csv' dosyasını yükleyin
# Kaggle linki: https://www.kaggle.com/datasets/danofer/sarcasm

from google.colab import files
print('Lütfen train-balanced-sarcasm.csv dosyasını yükleyin:')
uploaded = files.upload()
sarc_file = list(uploaded.keys())[0]
print(f'Yüklendi: {sarc_file}')

In [ ]:
# ─── 5. VERİ OKUMA VE İŞLEME ─────────────────────────────────

def load_semeval(path):
    """SemEval-2018 Task 3A formatı: index TAB label TAB tweet"""
    rows = []
    with open(path, encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i == 0:  # başlık satırı
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                label = int(parts[1])   # 0=non_irony, 1=irony
                text  = parts[2].strip()
                if text:
                    rows.append({'text': text, 'label': label})
    return pd.DataFrame(rows)

df_sem_train = load_semeval('semeval_train.txt')
df_sem_test  = load_semeval('semeval_test.txt')
df_semeval   = pd.concat([df_sem_train, df_sem_test], ignore_index=True)
print(f'SemEval: {len(df_semeval)} satır')
print(df_semeval['label'].value_counts())

In [ ]:
# SARC Reddit
df_sarc_raw = pd.read_csv(sarc_file)
print('SARC sütunları:', df_sarc_raw.columns.tolist())
print(df_sarc_raw.head(2))

In [ ]:
# SARC: text sütunu 'comment', etiket 'label' (1=sarcastic, 0=not)
df_sarc = df_sarc_raw[['comment', 'label']].dropna().copy()
df_sarc.columns = ['text', 'label']
df_sarc['label'] = df_sarc['label'].astype(int)
df_sarc['text']  = df_sarc['text'].astype(str).str.strip()
df_sarc = df_sarc[df_sarc['text'].str.len() > 5]

# Örneklem al (28K üst limit)
MAX_SARC = 28000
if len(df_sarc) > MAX_SARC:
    df_sarc = df_sarc.sample(MAX_SARC, random_state=SEED)

print(f'SARC: {len(df_sarc)} satır')
print(df_sarc['label'].value_counts())

In [ ]:
# Birleştir
df_all = pd.concat([df_semeval[['text','label']], df_sarc[['text','label']]], ignore_index=True)
df_all = df_all.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'\nToplam veri: {len(df_all)} satır')
print(f'İroni (1): {(df_all.label==1).sum()}')
print(f'Normal (0): {(df_all.label==0).sum()}')

# Train / Val / Test bölümü: %80 / %10 / %10
df_trainval, df_test = train_test_split(df_all, test_size=0.10, random_state=SEED, stratify=df_all['label'])
df_train, df_val    = train_test_split(df_trainval, test_size=0.111, random_state=SEED, stratify=df_trainval['label'])

print(f'\nEğitim: {len(df_train)} | Doğrulama: {len(df_val)} | Test: {len(df_test)}')

In [ ]:
# ─── 6. MODEL VE TOKENİZER ────────────────────────────────────
BASE_MODEL = 'roberta-base'   # Twitter içeriği için iyi temel model
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f'Tokenizer yüklendi: {BASE_MODEL}')

In [ ]:
# ─── 7. DATASET SINIFI ────────────────────────────────────────
class IronyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = IronyDataset(df_train['text'], df_train['label'], tokenizer, MAX_LEN)
val_dataset   = IronyDataset(df_val['text'],   df_val['label'],   tokenizer, MAX_LEN)
test_dataset  = IronyDataset(df_test['text'],  df_test['label'],  tokenizer, MAX_LEN)
print(f'Dataset hazır. Eğitim: {len(train_dataset)}')

In [ ]:
# ─── 8. MODEL ─────────────────────────────────────────────────
id2label = {0: 'non_irony', 1: 'irony'}
label2id = {'non_irony': 0, 'irony': 1}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)
model = model.to(device)
print('Model yüklendi.')

In [ ]:
# ─── 9. EĞİTİM PARAMETRELERİ ─────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1':       f1_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds)
    }

OUTPUT_DIR = 'toxicguard_irony_model'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Eğitim başlıyor...')

In [ ]:
# ─── 10. EĞİTİM ───────────────────────────────────────────────
trainer.train()

In [ ]:
# ─── 11. TEST DEĞERLENDİRME ───────────────────────────────────
print('\n=== TEST SETİ DEĞERLENDİRMESİ ===')
preds_out = trainer.predict(test_dataset)
y_pred    = np.argmax(preds_out.predictions, axis=-1)
y_true    = preds_out.label_ids

print(classification_report(y_true, y_pred, target_names=['non_irony', 'irony']))
f1 = f1_score(y_true, y_pred, average='macro')
acc = accuracy_score(y_true, y_pred)
print(f'F1-Macro: {f1:.4f}  |  Accuracy: {acc:.4f}')

In [ ]:
# ─── 12. MODELİ KAYDET ────────────────────────────────────────
SAVE_DIR = 'toxicguard_irony_model_final'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Eğitim bilgilerini kaydet
info = {
    'model_name': 'ToxicGuard-Irony',
    'version':    'V5.22-Stage2',
    'base_model': BASE_MODEL,
    'datasets':   ['SemEval-2018-Task3A', 'SARC-Reddit-Sarcasm'],
    'train_size': len(df_train),
    'val_size':   len(df_val),
    'test_size':  len(df_test),
    'f1_macro':   round(f1, 4),
    'accuracy':   round(acc, 4),
    'id2label':   id2label,
    'label2id':   label2id,
    'irony_threshold': 0.5
}

with open(f'{SAVE_DIR}/toxicguard_irony_info.json', 'w', encoding='utf-8') as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print(f'Model kaydedildi: {SAVE_DIR}/')
print(json.dumps(info, indent=2, ensure_ascii=False))

In [ ]:
# ─── 13. ZIP VE İNDİR ────────────────────────────────────────
import shutil
shutil.make_archive('toxicguard_irony_model_final', 'zip', SAVE_DIR)

from google.colab import files
files.download('toxicguard_irony_model_final.zip')
print('İndirme başladı!')

In [ ]:
# ─── 14. HIZLI TEST (opsiyonel) ───────────────────────────────
from transformers import pipeline

irony_pipe = pipeline(
    'text-classification',
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    top_k=None
)

test_sentences = [
    'Oh great, another Monday! Just what I needed.',
    'I love traffic jams, they make my day so much better.',
    'The weather is beautiful today!',
    'Sure, because breaking production on Friday is a great idea.',
    'I had a really nice time at the park.',
    'Yeah right, like that plan was ever going to work.',
]

print('\n=== HIZLI TEST ===')
for s in test_sentences:
    res = irony_pipe(s)[0]
    irony_score = next((r['score'] for r in res if r['label'] == 'irony'), 0)
    flag = '🎭' if irony_score >= 0.5 else '💬'
    print(f'{flag} [{irony_score:.3f}] {s}')